In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import os

%matplotlib inline

# ------------------------------------------------------------
# 1. Пътища / Paths
# ------------------------------------------------------------
input_dir = r"D:\data\master_thesis\input"
output_dir = r"D:\data\master_thesis\exports"
csv_path = os.path.join(input_dir, "fires_suggestion.csv")
north_arrow_path = os.path.join(input_dir, "north-arrow.png")   # локален файл / local file
os.makedirs(output_dir, exist_ok=True)

# ------------------------------------------------------------
# 2. Зареждане на данните / Load fire data
# ------------------------------------------------------------
df = pd.read_csv(csv_path, sep='\t')
if len(df.columns) == 1:
    df = pd.read_csv(csv_path, sep=',')
print(f"Заредени {len(df)} пожара. Колони: {df.columns.tolist()} / Loaded {len(df)} fires. Columns: {df.columns.tolist()}")

# ------------------------------------------------------------
# 3. Откриване на координатни колони / Detect coordinate columns
# ------------------------------------------------------------
cols_clean = {col.strip().lower(): col for col in df.columns}
lon_col = None
for key in ['lon', 'longitude', 'long']:
    if key in cols_clean:
        lon_col = cols_clean[key]
        break
lat_col = None
for key in ['lat', 'latitude']:
    if key in cols_clean:
        lat_col = cols_clean[key]
        break
if not lon_col or not lat_col:
    raise KeyError(f"Не са открити координатни колони / Coordinate columns missing. Налични: {list(df.columns)}")
print(f"Използвани координатни колони: {lon_col}, {lat_col} / Using: {lon_col}, {lat_col}")

gdf_fires = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
    crs="EPSG:4326"
)

# ------------------------------------------------------------
# 4. Граница на България / Bulgaria boundary
# ------------------------------------------------------------
url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)
bulgaria = world[world['NAME'] == 'Bulgaria']

# ------------------------------------------------------------
# 5. Проекция в Web Mercator / Reproject to Web Mercator
# ------------------------------------------------------------
bulgaria = bulgaria.to_crs(epsg=3857)
gdf_fires = gdf_fires.to_crs(epsg=3857)

# ------------------------------------------------------------
# 6. Карта – цялата територия на България / Map – whole country
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 12))

# По-големи точки / Bigger dots
gdf_fires.plot(ax=ax, color='red', markersize=100, alpha=0.8, label='Пожар')

# Етикети с номера / Labels with Fire_id
if 'Fire_id' in df.columns:
    for idx, row in gdf_fires.iterrows():
        ax.annotate(
            str(row['Fire_id']),
            (row.geometry.x, row.geometry.y),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=7,
            color='darkred'
        )

# Задаване на обхвата спрямо границите на България (с 10% отстъп)
# Set map extent to Bulgaria's bounding box (with 10% margin)
bounds = bulgaria.total_bounds          # [minx, miny, maxx, maxy]
margin_x = (bounds[2] - bounds[0]) * 0.1
margin_y = (bounds[3] - bounds[1]) * 0.1
ax.set_xlim(bounds[0] - margin_x, bounds[2] + margin_x)
ax.set_ylim(bounds[1] - margin_y, bounds[3] + margin_y)

# ------------------------------------------------------------
# 7. Фонова карта / Basemap
# ------------------------------------------------------------
try:
    import contextily as ctx
    ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik, zoom=8)
    print("Фоновата карта е добавена. / Basemap added.")
except ImportError:
    print("contextily не е инсталиран – само контурна карта. / contextily not installed – outline map only.")

# ------------------------------------------------------------
# 8. Северна стрелка (локално изображение) / North arrow (local image)
# ------------------------------------------------------------
north_img = plt.imread(north_arrow_path)
imagebox = OffsetImage(north_img, zoom=0.15)
ab = AnnotationBbox(
    imagebox,
    (1, 1),                       # горе вдясно / top-right
    xycoords='axes fraction',
    box_alignment=(1, 1),
    frameon=False
)
ax.add_artist(ab)

# ------------------------------------------------------------
# 9. Мащабна линия / Scale bar
# ------------------------------------------------------------
try:
    from matplotlib_scalebar.scalebar import ScaleBar
    scalebar = ScaleBar(
        1,
        units='m',
        dimension='si-length',
        location='lower right',
        scale_loc='bottom',
        label_loc='top',
        frameon=False,
        color='black',
        box_alpha=0,
        font_properties={'size': 10},
        fixed_units='km',
        fixed_value=50,
    )
    ax.add_artist(scalebar)
    print("Мащабна линия е добавена (matplotlib-scalebar). / Scalebar added (matplotlib-scalebar).")
except ImportError:
    print("matplotlib-scalebar не е инсталиран – създаване на ръчна скала. / matplotlib-scalebar not installed – creating manual scale bar.")
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    bar_length = 50000
    bar_x_start = xlim[1] - bar_length - 15000
    bar_y = ylim[0] + 30000
    ax.plot([bar_x_start, bar_x_start + bar_length], [bar_y, bar_y],
            color='black', linewidth=3)
    ax.text(bar_x_start + bar_length/2, bar_y - 15000, '50 km',
            ha='center', va='top', fontsize=9, color='black',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.7))
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

# ------------------------------------------------------------
# 10. Заглавие, легенда (горе вляво, за да не се застъпва), запис
#    Title, legend (upper left to avoid overlap), save
# ------------------------------------------------------------
ax.set_axis_off()
ax.set_title("Точкова карта на извадка от пожари в България през 2025г.",
             fontsize=14, pad=20)
# Легендата е преместена горе вляво / Legend moved to upper left
ax.legend(loc='upper left', framealpha=0.9)
plt.tight_layout()
plt.show()

output_map = os.path.join(output_dir, "fires_dot_map.png")
fig.savefig(output_map, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Картата е запазена в: {output_map} / Map saved to: {output_map}")